# Generalization Study: Unseen-Cell & Temporal Extrapolation on NASA Battery Benchmark
## TE-Q-Transformer Research Framework

**Scientific Research Question:**  
*"Does the model generalize to strictly unseen cells across distinct operating temperatures and to forward temporal horizons under prolonged degradation?"*

---

### Evaluation Protocols:
1. **Unseen-Cell Generalization:** Cells `B0018` (132 cycles, 24°C) and `B0032` (39 cycles, 4°C) held out completely from training and scaler fitting.
2. **Temporal Extrapolation:** Cell `B0053` (44°C) with cycles 0–36 used in training and cycles 37–52 evaluated as forward temporal forecasting.

In [ ]:
# ==============================================================================
# 0. SETUP ENVIRONMENT AND REPOSITORY PATHS
# ==============================================================================
import os
import sys
from pathlib import Path

MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / "TE-Q-Transformer",
    Path("/kaggle/working") / "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation",
    Path("/kaggle/working"),
    Path("/content") / "TE-Q-Transformer",
    Path("/content") / "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation",
    Path("/content"),
]

# Auto-discover any Kaggle input mount points
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("*"):
        CANDIDATES.extend([p, p / "TE-Q-Transformer"])
        for sub in p.glob("*"):
            if sub.is_dir():
                CANDIDATES.append(sub)

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
# Dynamic data root resolution across local and Kaggle mounts
data_candidates = [
    REPO_ROOT / "datasets",
    Path("/kaggle/working/TE-Q-Transformer/datasets"),
    Path("/kaggle/working/datasets"),
    Path("datasets"),
]
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("**/NASA/processed"):
        data_candidates.insert(0, p.parent.parent)
DATA_ROOT = next((p.resolve() for p in data_candidates if p.exists() and (p / "NASA" / "processed").exists()), (REPO_ROOT / "datasets").resolve())
MODEL_ROOT = REPO_ROOT / "models"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")

## 1. NASA Ames Dataset Preparation
Loading the NASA dataset and isolating held-out test splits.

In [ ]:
from models.proposed import TEQTransformer, rich_entangler_config
from src.eval.metrics import calculate_metrics, calculate_macro_metrics

def evaluate_nasa_generalization(model: nn.Module, test_loaders: dict, device: torch.device) -> dict:
    model.eval()
    model.to(device)
    per_cell = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            preds, actuals = [], []
            for bx, by in loader:
                bx = bx.to(device)
                preds.append(model(bx).cpu().numpy())
                actuals.append(by.numpy())
            y_pred = np.concatenate(preds)
            y_true = np.concatenate(actuals)
            per_cell[cell_id] = {
                "metrics": calculate_metrics(y_true, y_pred),
                "preds": y_pred,
                "actuals": y_true,
            }
    macro = calculate_macro_metrics([res["metrics"] for res in per_cell.values()])
    return {"per_cell": per_cell, "macro": macro}

# Checkpoint resolution
ckpt_candidates = [
    REPO_ROOT / "models" / "checkpoints" / "nasa_teq_transformer_best.pth",
    REPO_ROOT / "artifacts" / "nasa" / "01_rich_entangler" / "nasa_teq_transformer_best.pth",
]
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("**/nasa_teq_transformer_best.pth"):
        ckpt_candidates.insert(0, p)

ckpt_path = next((p.resolve() for p in ckpt_candidates if p.exists()), None)
model = TEQTransformer(rich_entangler_config()).to(DEVICE)

if ckpt_path:
    print(f"[Setup] Loading reference weights from: {ckpt_path}")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
else:
    print("[Notice] Checkpoint not found. Running demonstration with standard initialized model.")

gen_results = evaluate_nasa_generalization(model, test_loaders, DEVICE)

print("\nNASA Generalization Results:")
for cell_id, cdata in gen_results["per_cell"].items():
    m = cdata["metrics"]
    print(f"  - Cell {cell_id:<12}: RMSE={m['RMSE']:.5f} | MAE={m['MAE']:.5f} | R2={m['R2']:.5f}")
print(f"\n  >> Macro Average: RMSE={gen_results['macro']['RMSE']:.5f} | MAE={gen_results['macro']['MAE']:.5f} | R2={gen_results['macro']['R2']:.5f}")


## 3. Degradation Trajectory Visualization
Plotting ground truth vs predicted State-of-Health degradation trajectories across the held-out test cells (`B0018`, `B0032`, `B0053_test`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), dpi=120)
for ax, (cell_id, cdata) in zip(axes, gen_results["per_cell"].items()):
    y_true = cdata["actuals"]
    y_pred = cdata["preds"]
    cycles = np.arange(1, len(y_true) + 1)
    m = cdata["metrics"]
    ax.plot(cycles, y_true, 'k-', lw=2, label="Actual SOH")
    ax.plot(cycles, y_pred, 'r--', lw=2, label="TE-Q-Transformer")
    ax.set_title(f"Cell {cell_id} (RMSE: {m['RMSE']:.4f}, R²: {m['R2']:.3f})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Cycle Index", fontsize=10)
    ax.set_ylabel("SOH", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(frameon=True, facecolor="white", framealpha=0.9)

plt.tight_layout()
plt.show()
